In [2]:
import os
from dotenv import load_dotenv
import musicbrainzngs
import pandas as pd
import time

load_dotenv()
musicbrainzngs.set_useragent(
    os.getenv("MB_USER_AGENT"),
    os.getenv("MB_VERSION"),
    os.getenv("MB_EMAIL")
)

In [7]:
def fetch_music_data(target_count_per_genre=100):
    genres = ['rock', 'electronic', 'jazz', 'hip hop', 'pop']
    all_artists = []

    for genre in genres:
        print(f"Fetching data for: {genre}")
        offset = 0
        genre_found = 0

        while genre_found < target_count_per_genre:
            try:
                search_result = musicbrainzngs.search_artists(
                    tag = genre,
                    limit = 100,
                    offset = offset
                )

                for artist in search_result['artist-list']:
                    artist_id = artist['id']
                    details = musicbrainzngs.get_artist_by_id(
                        artist_id,
                        includes = ["release-groups", "tags", "release-rels", "label-rels"]
                    )

                    artist_info = details['artist']
                    rg_list = artist_info.get('release-group-list', [])
                    albums = [rg for rg in rg_list if rg.get("type") == 'Album']

                    label_rels = artist_info.get('label-relation-list', [])
                    label_count = len(label_rels)

                    release_rels = artist_info.get('release-relation-list', [])
                    rel_count = len(release_rels)

                    all_artists.append({
                        'mbid': artist_id,
                        'name': artist_info.get('name'),
                        'type': artist_info.get('type'),
                        'gender': artist_info.get('gender'),
                        'area_name': artist_info.get('area', {}).get('name'),
                        'begin_date': artist_info.get('life-span', {}).get('begin'),
                        'end_date': artist_info.get('life-span', {}).get('end'),
                        'ended': artist_info.get('life-span', {}).get('ended'),
                        'tags': [t['name'] for t in artist_info.get('tag-list', [])],
                        'genre_label': genre,
                        'label_count': label_count,
                        'release_rel_count': rel_count,
                        'album_count': len(albums)
                    })

                    genre_found += 1
                    if genre_found % 10 == 0:
                        print(f"Downloaded {genre_found} / {target_count_per_genre} for {genre}")

                    time.sleep(1.1)

                    if genre_found >= target_count_per_genre:
                        break

                offset += 100
                if offset > 1000:
                    break

            except Exception as e:
                print(f"Error fetching {genre}: {repr(e)}")
                time.sleep(5)
                continue

    return pd.DataFrame(all_artists)

In [8]:
df_raw = fetch_music_data(target_count_per_genre=400)
df_raw.to_csv("artists_data_raw.csv", index = False, encoding = "utf-8")
print(f"Downloaded {len(df_raw)} artists data")

Fetching data for: rock
Downloaded 10 / 400 for rock
Downloaded 20 / 400 for rock
Downloaded 30 / 400 for rock
Downloaded 40 / 400 for rock
Downloaded 50 / 400 for rock
Downloaded 60 / 400 for rock
Downloaded 70 / 400 for rock
Downloaded 80 / 400 for rock
Downloaded 90 / 400 for rock
Downloaded 100 / 400 for rock
Downloaded 110 / 400 for rock
Downloaded 120 / 400 for rock
Downloaded 130 / 400 for rock
Downloaded 140 / 400 for rock
Downloaded 150 / 400 for rock
Downloaded 160 / 400 for rock
Downloaded 170 / 400 for rock
Downloaded 180 / 400 for rock
Downloaded 190 / 400 for rock
Downloaded 200 / 400 for rock
Downloaded 210 / 400 for rock
Downloaded 220 / 400 for rock
Downloaded 230 / 400 for rock
Downloaded 240 / 400 for rock
Downloaded 250 / 400 for rock
Downloaded 260 / 400 for rock
Downloaded 270 / 400 for rock
Downloaded 280 / 400 for rock
Downloaded 290 / 400 for rock
Downloaded 300 / 400 for rock
Downloaded 310 / 400 for rock
Downloaded 320 / 400 for rock
Downloaded 330 / 400 for 

In [10]:
df_raw.head()

,mbid,name,type,gender,area_name,begin_date,end_date,ended,tags,genre_label,label_count,release_rel_count,album_count
0,89ad4ac3-39f7-470e-963a-56509c546377,Various Artists,Other,NaN,NaN,NaN,NaN,NaN,"[compilation, country dreams, special purpose,...",rock,4,3,0
1,70248960-cb53-4ea4-943a-edb18f7d336f,Bruce Springsteen,Person,Male,United States,1949-09-23,NaN,NaN,"[american, americana, aor, big music, contempo...",rock,1,1511,21
2,b10bbbfc-cf9e-42e0-be17-e2c3e1d2600d,The Beatles,Group,NaN,United Kingdom,1960,1970-04-10,true,"[60s, beat, british, british invasion, britpop...",rock,2,0,20
3,d6652e7b-33fe-49ef-8336-4c863b4f996f,The E Street Band,Group,NaN,United States,1972-10,NaN,NaN,"[heartland rock, pop rock, rock]",rock,0,0,0
4,b071f9fa-14b0-4217-8e97-eb41da73f598,The Rolling Stones,Group,NaN,United Kingdom,1962,NaN,NaN,"[blues, blues rock, british, british rhythm & ...",rock,2,8,25
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1990,3c004c98-aab6-4b63-a2df-e07c98e73b0a,José Feliciano,Person,Male,United States,1945-09-10,NaN,NaN,"[acoustic guitar, bolero, christmas music, fol...",pop,0,141,24
1991,e449337e-4192-415d-88d0-81352c068d2c,Brook Benton,Person,Male,United States,1931-09-19,1988-04-09,true,"[country soul, pop, r&b, rhythm & blues, rhyth...",pop,0,0,25
1992,1bebb19e-9305-4da8-a3fd-5dd40a6e517e,Mahalia Jackson,Person,Female,United States,1911-10-26,1972-01-27,true,"[blues, classic pop and rock, gospel, jazz]",pop,0,1,25
1993,b134d1bf-c7c7-4427-93ac-9fdbc2b59ef1,Meat Loaf,Person,Male,United States,1947-09-27,2022-01-20,true,"[2008 universal fire victim, aor, classic rock...",pop,1,0,13
